# VLM Disaster Evaluation — All 4 Models (T4 GPU)

Evaluates 300 extracted frames (75 videos × 4 frames) against:
- **CLIP** — zero-shot classification (`openai/clip-vit-base-patch32`)
- **BLIP-2** — VQA captioning (`Salesforce/blip2-opt-2.7b`)
- **LLaVA** — structured scene analysis (`llava-hf/llava-1.5-7b-hf`)
- **Qwen** — structured disaster analysis (`Qwen/Qwen2-VL-2B-Instruct`)

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `colab_frames.zip` to your Google Drive (run `python scripts/video_pipeline/package_for_colab.py` locally to generate it)
3. Set `DRIVE_ZIP_PATH` in Cell 3 to match where you placed the zip in Drive

In [ ]:
# ── Cell 1: GPU check + install dependencies ─────────────────────────────────
import subprocess, sys

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Switch to T4 runtime before continuing.")

# Install required packages
!pip install -q transformers==4.45.2 accelerate bitsandbytes sentencepiece Pillow openpyxl tqdm pandas
!pip install -q qwen-vl-utils
print("\nAll dependencies installed.")

In [ ]:
# ── Cell 2: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

In [ ]:
# ── Cell 3: Extract frames dataset ──────────────────────────────────────────
# !! UPDATE THIS PATH to where you placed colab_frames.zip in your Drive !!
DRIVE_ZIP_PATH = "/content/drive/MyDrive/colab_frames.zip"

import zipfile, os
from pathlib import Path

WORK_DIR = Path("/content/vlm_eval")
WORK_DIR.mkdir(exist_ok=True)

zip_path = Path(DRIVE_ZIP_PATH)
if not zip_path.exists():
    raise FileNotFoundError(
        f"Zip not found at {DRIVE_ZIP_PATH}\n"
        "Run: python scripts/video_pipeline/package_for_colab.py  (locally)\n"
        "Then upload the resulting colab_frames.zip to your Google Drive."
    )

print(f"Extracting {zip_path.name} ({zip_path.stat().st_size / 1e6:.1f} MB)...")
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(WORK_DIR)

FRAMES_ROOT  = WORK_DIR / "extracted_frames"
MANIFEST_CSV = WORK_DIR / "frame_manifest.csv"
RESULTS_DIR  = WORK_DIR / "evaluation"
RESULTS_DIR.mkdir(exist_ok=True)

frame_count = len(list(FRAMES_ROOT.rglob("*.jpg")))
print(f"Frames extracted : {frame_count}")
print(f"Manifest         : {MANIFEST_CSV.exists()}")
print(f"Results dir      : {RESULTS_DIR}")

import pandas as pd
frame_df_global = pd.read_csv(MANIFEST_CSV)
print(f"Manifest rows    : {len(frame_df_global)}")
print(frame_df_global.head(3))

In [ ]:
# ── Cell 4: Shared utilities ─────────────────────────────────────────────────
import io, gc, logging, torch
from collections import Counter
from pathlib import Path
from PIL import Image
import pandas as pd
from tqdm.notebook import tqdm

logging.basicConfig(level=logging.WARNING)  # suppress debug noise

# ── mirrors src/utils/metrics.py ─────────────────────────────────────────────
def confidence_to_level(score: float) -> str:
    if score > 88: return "Critical"
    if score > 75: return "High"
    if score > 60: return "Moderate"
    return "Low"

# ── mirrors evaluate_videos.py ────────────────────────────────────────────────
_CATEGORY_ALIASES = {
    "flood":      {"water disaster", "flood", "flooded"},
    "wildfire":   {"wild fire", "wildfire", "fire", "on fire", "urban fire"},
    "earthquake": {"earthquake", "infrastructure damage"},
    "landslide":  {"landslide", "mudslide"},
    "cyclone":    {"cyclone", "water disaster", "tropical cyclone", "tornado"},
}

def _is_correct(prediction: str, category: str) -> bool:
    pred_lower = (prediction or "").lower()
    aliases    = _CATEGORY_ALIASES.get(category, {category})
    return any(alias in pred_lower for alias in aliases)

def aggregate_per_video(frame_df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for (video_stem, category), grp in frame_df.groupby(["video_stem", "category"]):
        preds    = [p for p in grp["prediction"].tolist()
                    if p and p not in ("MISSING", "N/A", "")]
        majority = Counter(preds).most_common(1)[0][0] if preds else "N/A"
        avg_conf = grp["confidence"].mean()
        model_name = grp["model"].iloc[0] if not grp.empty else "unknown"
        records.append({
            "video_stem":          video_stem,
            "category":            category,
            "model":               model_name,
            "frames_evaluated":    len(grp),
            "majority_prediction": majority,
            "avg_confidence":      round(float(avg_conf), 2) if pd.notna(avg_conf) else None,
            "correct":             _is_correct(majority, category),
        })
    return pd.DataFrame(records)

def _to_pil(frame_path: Path) -> Image.Image:
    return Image.open(frame_path).convert("RGB")

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def save_model_results(frame_records: list, model_name: str) -> pd.DataFrame:
    frame_df  = pd.DataFrame(frame_records)
    video_df  = aggregate_per_video(frame_df)
    frame_df.to_csv(RESULTS_DIR / f"{model_name}_frame_results.csv", index=False)
    video_df.to_csv(RESULTS_DIR / f"{model_name}_results.csv", index=False)
    n_correct = video_df["correct"].sum()
    n_total   = len(video_df)
    acc       = n_correct / n_total * 100 if n_total else 0
    print(f"  [{model_name.upper():5}] Accuracy: {acc:.1f}%  ({n_correct}/{n_total} videos correct)")
    return video_df

print("Shared utilities loaded.")

In [ ]:
# ── Cell 5: CLIP evaluation ──────────────────────────────────────────────────
from transformers import CLIPProcessor, CLIPModel

CLIP_MODEL_PATH = "openai/clip-vit-base-patch32"

CLIP_PROMPTS = [
    "an image showing earthquake damage with collapsed buildings and rubble",
    "an image showing damaged infrastructure such as roads, bridges, or buildings",
    "an image showing injured or affected people after a disaster",
    "an image of a wildfire burning vegetation and forests",
    "an image of an urban fire affecting buildings and city areas",
    "an image showing flood or water disaster with submerged areas",
    "an image showing drought conditions with dry cracked land",
    "an image showing a landslide with collapsed terrain and debris",
    "an image of a forest without disaster",
    "an image of buildings and streets without disaster",
    "an image of the sea or ocean",
    "an image showing people in normal conditions"
]
CLIP_LABELS = [
    "Earthquake", "Infrastructure Damage", "Human Damage", "Wild Fire", "Urban Fire",
    "Water Disaster", "Drought", "Landslide", "Forest", "Buildings and Street",
    "Sea", "Human"
]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading CLIP on {device}...")
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_PATH)
clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_PATH).to(device)
clip_model.eval()

clip_records = []
for _, row in tqdm(frame_df_global.iterrows(), total=len(frame_df_global), desc="CLIP"):
    fp = FRAMES_ROOT / Path(row["frame_path"]).relative_to(Path(row["frame_path"]).parts[0]) \
         if not Path(str(row["frame_path"])).is_absolute() \
         else Path(str(row["frame_path"]))
    # Build the path relative to the extracted frames root
    cat       = row["category"]
    vstem     = row["video_stem"]
    fname     = row["frame_filename"] if "frame_filename" in row else Path(row["frame_path"]).name
    fp        = FRAMES_ROOT / cat / vstem / fname

    if not fp.exists():
        clip_records.append({"video_stem": vstem, "category": cat,
                              "frame_path": str(fp), "frame_number": row.get("frame_number"),
                              "model": "clip", "prediction": "MISSING",
                              "confidence": None, "raw_response": "", "error": "file not found"})
        continue

    try:
        image  = _to_pil(fp)
        inputs = clip_processor(text=CLIP_PROMPTS, images=image,
                                return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = clip_model(**inputs)
        probs          = outputs.logits_per_image.softmax(dim=1)[0]
        pred_idx       = probs.argmax().item()
        prediction     = CLIP_LABELS[pred_idx]
        confidence     = round(probs[pred_idx].item() * 100, 2)
        clip_records.append({"video_stem": vstem, "category": cat,
                              "frame_path": str(fp), "frame_number": row.get("frame_number"),
                              "model": "clip", "prediction": prediction,
                              "confidence": confidence, "raw_response": prediction, "error": ""})
    except Exception as e:
        clip_records.append({"video_stem": vstem, "category": cat,
                              "frame_path": str(fp), "frame_number": row.get("frame_number"),
                              "model": "clip", "prediction": "N/A",
                              "confidence": None, "raw_response": "", "error": str(e)})

clip_video_df = save_model_results(clip_records, "clip")

# Free CLIP before loading next model
del clip_model, clip_processor
free_gpu()
print("CLIP done. GPU freed.")

In [ ]:
# ── Cell 6: BLIP-2 evaluation ────────────────────────────────────────────────
from transformers import Blip2Processor, Blip2ForConditionalGeneration

BLIP2_MODEL_PATH = "Salesforce/blip2-opt-2.7b"
BLIP2_PROMPT = (
    "Question: What natural disaster is shown in this image "
    "and what damage or environmental impact is visible? "
    "Answer:"
)
BLIP2_MAX_NEW_TOKENS = 120

STOPWORDS = {
    "that", "this", "with", "from", "have", "been", "were", "they",
    "their", "there", "which", "where", "what", "when", "will", "about",
    "also", "into", "some", "such", "than", "then", "these", "those",
    "both", "each", "more", "most", "other", "over", "under", "after",
    "before", "while", "image", "shows", "shown", "visible", "area",
    "areas", "large", "small", "many", "several",
}

def _extract_keywords(caption: str) -> list:
    seen, keywords = set(), []
    for word in caption.lower().split():
        clean = word.strip(".,!?;:\"'()-")
        if len(clean) > 4 and clean not in STOPWORDS and clean not in seen:
            seen.add(clean); keywords.append(clean)
    return keywords[:5]

print(f"Loading BLIP-2 on {device} (fp16, ~5.4 GB VRAM)...")
blip2_processor = Blip2Processor.from_pretrained(BLIP2_MODEL_PATH)
blip2_model     = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_MODEL_PATH, torch_dtype=torch.float16, device_map="auto"
)
blip2_model.eval()
print(f"BLIP-2 loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

blip2_records = []
for _, row in tqdm(frame_df_global.iterrows(), total=len(frame_df_global), desc="BLIP-2"):
    cat   = row["category"]
    vstem = row["video_stem"]
    fname = row["frame_filename"] if "frame_filename" in row else Path(row["frame_path"]).name
    fp    = FRAMES_ROOT / cat / vstem / fname

    if not fp.exists():
        blip2_records.append({"video_stem": vstem, "category": cat,
                               "frame_path": str(fp), "frame_number": row.get("frame_number"),
                               "model": "blip2", "prediction": "MISSING",
                               "confidence": None, "raw_response": "", "error": "file not found"})
        continue

    try:
        image  = _to_pil(fp)
        inputs = blip2_processor(text=BLIP2_PROMPT, images=image,
                                  return_tensors="pt").to(device, torch.float16)
        with torch.no_grad():
            generated = blip2_model.generate(
                **inputs, max_new_tokens=BLIP2_MAX_NEW_TOKENS,
                output_scores=True, return_dict_in_generate=True
            )
        trimmed = generated.sequences[:, inputs["input_ids"].shape[1]:]
        caption = blip2_processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
        token_probs = [torch.softmax(s.float(), dim=-1).max().item() for s in generated.scores]
        confidence  = round(sum(token_probs) / len(token_probs) * 100, 2) if token_probs else 0.0
        blip2_records.append({"video_stem": vstem, "category": cat,
                               "frame_path": str(fp), "frame_number": row.get("frame_number"),
                               "model": "blip2", "prediction": caption[:80],
                               "confidence": confidence, "raw_response": caption[:200], "error": ""})
    except Exception as e:
        blip2_records.append({"video_stem": vstem, "category": cat,
                               "frame_path": str(fp), "frame_number": row.get("frame_number"),
                               "model": "blip2", "prediction": "N/A",
                               "confidence": None, "raw_response": "", "error": str(e)})

blip2_video_df = save_model_results(blip2_records, "blip2")

del blip2_model, blip2_processor
free_gpu()
print("BLIP-2 done. GPU freed.")

In [ ]:
# ── Cell 7: LLaVA evaluation ─────────────────────────────────────────────────
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

LLAVA_MODEL_PATH     = "llava-hf/llava-1.5-7b-hf"
LLAVA_MAX_NEW_TOKENS = 200
LLAVA_PROMPT = (
    "USER: <image>\n"
    "Analyze this disaster image and respond ONLY in this exact format:\n"
    "DISASTER TYPE: [type]\n"
    "SEVERITY: [Critical/High/Moderate/Low]\n"
    "AFFECTED AREA: [description]\n"
    "INFRASTRUCTURE DAMAGE: [Yes/No - description]\n"
    "RECOMMENDED ACTION: [action]\n"
    "ASSISTANT:"
)
LLAVA_FIELD_MAP = {
    "DISASTER TYPE":         "disaster_type",
    "SEVERITY":              "severity",
    "AFFECTED AREA":         "affected_areas",
    "INFRASTRUCTURE DAMAGE": "infrastructure_damage",
    "RECOMMENDED ACTION":    "recommended_action",
}

def _parse_llava(text: str) -> dict:
    fields = {v: "" for v in LLAVA_FIELD_MAP.values()}
    for line in text.splitlines():
        for label, key in LLAVA_FIELD_MAP.items():
            if line.startswith(label + ":"):
                fields[key] = line[len(label) + 1:].strip(); break
    return fields

# 4-bit quantization so LLaVA-1.5-7B fits comfortably in 16 GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

print("Loading LLaVA-1.5-7B (4-bit NF4, ~4 GB VRAM)...")
llava_processor = AutoProcessor.from_pretrained(LLAVA_MODEL_PATH)
llava_model     = LlavaForConditionalGeneration.from_pretrained(
    LLAVA_MODEL_PATH, quantization_config=bnb_config, device_map="auto",
    low_cpu_mem_usage=True,
)
llava_model.eval()
print(f"LLaVA loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

llava_records = []
for _, row in tqdm(frame_df_global.iterrows(), total=len(frame_df_global), desc="LLaVA"):
    cat   = row["category"]
    vstem = row["video_stem"]
    fname = row["frame_filename"] if "frame_filename" in row else Path(row["frame_path"]).name
    fp    = FRAMES_ROOT / cat / vstem / fname

    if not fp.exists():
        llava_records.append({"video_stem": vstem, "category": cat,
                               "frame_path": str(fp), "frame_number": row.get("frame_number"),
                               "model": "llava", "prediction": "MISSING",
                               "confidence": None, "raw_response": "", "error": "file not found"})
        continue

    try:
        image  = _to_pil(fp)
        inputs = llava_processor(text=LLAVA_PROMPT, images=image, return_tensors="pt")
        inputs = {k: v.to(llava_model.device) for k, v in inputs.items()}
        with torch.no_grad():
            output = llava_model.generate(
                **inputs, max_new_tokens=LLAVA_MAX_NEW_TOKENS,
                output_scores=True, return_dict_in_generate=True
            )
        full_text = llava_processor.decode(output.sequences[0], skip_special_tokens=True)
        response  = full_text.split("ASSISTANT:")[-1].strip() if "ASSISTANT:" in full_text else full_text.strip()
        token_probs  = [torch.softmax(s.float(), dim=-1).max().item() for s in output.scores]
        confidence   = round(sum(token_probs) / len(token_probs) * 100, 2) if token_probs else 0.0
        fields       = _parse_llava(response)
        disaster_type = fields.get("disaster_type", "") or response[:80]
        llava_records.append({"video_stem": vstem, "category": cat,
                               "frame_path": str(fp), "frame_number": row.get("frame_number"),
                               "model": "llava", "prediction": disaster_type,
                               "confidence": confidence, "raw_response": response[:200], "error": ""})
    except Exception as e:
        llava_records.append({"video_stem": vstem, "category": cat,
                               "frame_path": str(fp), "frame_number": row.get("frame_number"),
                               "model": "llava", "prediction": "N/A",
                               "confidence": None, "raw_response": "", "error": str(e)})

llava_video_df = save_model_results(llava_records, "llava")

del llava_model, llava_processor
free_gpu()
print("LLaVA done. GPU freed.")

In [ ]:
# ── Cell 8: Qwen evaluation ──────────────────────────────────────────────────
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

QWEN_MODEL_PATH     = "Qwen/Qwen2-VL-2B-Instruct"
QWEN_MAX_NEW_TOKENS = 1024
QWEN_PROMPT = (
    "Analyze this disaster image.\n"
    "Return ONLY in this exact format:\n"
    "DISASTER TYPE:\n"
    "SEVERITY:\n"
    "ONE-LINE DESCRIPTION:"
)
QWEN_FIELD_MAP = {
    "DISASTER TYPE":       "disaster_type",
    "SEVERITY":            "severity",
    "AFFECTED POPULATION": "affected_population",
    "INFRASTRUCTURE":      "infrastructure_status",
    "ENVIRONMENT":         "environmental_impact",
}

def _parse_qwen(text: str) -> dict:
    fields = {v: "" for v in QWEN_FIELD_MAP.values()}
    for line in text.splitlines():
        stripped = line.strip()
        for label, key in QWEN_FIELD_MAP.items():
            if stripped.startswith(label + ":"):
                fields[key] = stripped[len(label) + 1:].strip(); break
    return fields

print(f"Loading Qwen2-VL-2B on {device} (fp16, ~4 GB VRAM)...")
qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_PATH, trust_remote_code=True)
qwen_model     = Qwen2VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_PATH, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True,
)
qwen_model.eval()
print(f"Qwen loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

qwen_records = []
for _, row in tqdm(frame_df_global.iterrows(), total=len(frame_df_global), desc="Qwen"):
    cat   = row["category"]
    vstem = row["video_stem"]
    fname = row["frame_filename"] if "frame_filename" in row else Path(row["frame_path"]).name
    fp    = FRAMES_ROOT / cat / vstem / fname

    if not fp.exists():
        qwen_records.append({"video_stem": vstem, "category": cat,
                              "frame_path": str(fp), "frame_number": row.get("frame_number"),
                              "model": "qwen", "prediction": "MISSING",
                              "confidence": None, "raw_response": "", "error": "file not found"})
        continue

    try:
        image    = _to_pil(fp)
        messages = [{"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text":  QWEN_PROMPT},
        ]}]
        text   = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = qwen_processor(text=[text], images=[image], return_tensors="pt")
        inputs = {k: v.to(qwen_model.device) for k, v in inputs.items()}
        with torch.no_grad():
            generated = qwen_model.generate(
                **inputs, max_new_tokens=QWEN_MAX_NEW_TOKENS,
                do_sample=False, num_beams=1, use_cache=True,
                output_scores=True, return_dict_in_generate=True,
            )
        trimmed  = [out[len(inp):] for inp, out in zip(inputs["input_ids"], generated.sequences)]
        response = qwen_processor.batch_decode(trimmed, skip_special_tokens=True,
                                               clean_up_tokenization_spaces=False)[0].strip()
        token_probs   = [torch.softmax(s.float(), dim=-1).max().item() for s in generated.scores]
        confidence    = round(sum(token_probs) / len(token_probs) * 100, 2) if token_probs else 0.0
        fields        = _parse_qwen(response)
        disaster_type = fields.get("disaster_type", "") or response[:80]
        qwen_records.append({"video_stem": vstem, "category": cat,
                              "frame_path": str(fp), "frame_number": row.get("frame_number"),
                              "model": "qwen", "prediction": disaster_type,
                              "confidence": confidence, "raw_response": response[:200], "error": ""})
    except Exception as e:
        qwen_records.append({"video_stem": vstem, "category": cat,
                              "frame_path": str(fp), "frame_number": row.get("frame_number"),
                              "model": "qwen", "prediction": "N/A",
                              "confidence": None, "raw_response": "", "error": str(e)})

qwen_video_df = save_model_results(qwen_records, "qwen")

del qwen_model, qwen_processor
free_gpu()
print("Qwen done. GPU freed.")

In [ ]:
# ── Cell 9: Export Video_VLM_Comparison.xlsx + copy to Drive ─────────────────
import openpyxl
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, PatternFill, Alignment

HEADER_BG  = "2F4F7F"
HEADER_FG  = "FFFFFF"
CORRECT_BG = "C6EFCE"
WRONG_BG   = "FFC7CE"

fill_h   = PatternFill("solid", fgColor=HEADER_BG)
font_h   = Font(color=HEADER_FG, bold=True)
fill_ok  = PatternFill("solid", fgColor=CORRECT_BG)
fill_bad = PatternFill("solid", fgColor=WRONG_BG)
align_c  = Alignment(horizontal="center")

def _style(ws, df):
    for cell in ws[1]:
        cell.fill = fill_h; cell.font = font_h; cell.alignment = align_c
    for ci, col in enumerate(df.columns, start=1):
        max_len = max(len(str(col)),
                      df[col].astype(str).str.len().max() if len(df) > 0 else 0)
        ws.column_dimensions[get_column_letter(ci)].width = min(max_len + 4, 50)

# Collect all model results (handle if a model failed)
model_video_dfs = {}
for name, var in [("clip", "clip_video_df"), ("blip2", "blip2_video_df"),
                  ("llava", "llava_video_df"), ("qwen", "qwen_video_df")]:
    csv_path = RESULTS_DIR / f"{name}_results.csv"
    if csv_path.exists():
        model_video_dfs[name] = pd.read_csv(csv_path)
    else:
        print(f"  WARNING: {name}_results.csv not found — skipping")

out_xlsx = RESULTS_DIR / "Video_VLM_Comparison.xlsx"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    # Per-model sheets
    for model, df in model_video_dfs.items():
        sname = model.upper()
        df.to_excel(writer, sheet_name=sname, index=False)
        ws = writer.sheets[sname]
        _style(ws, df)
        if "correct" in df.columns:
            c_col = list(df.columns).index("correct") + 1
            for ri, correct in enumerate(df["correct"], start=2):
                ws.cell(ri, c_col).fill = fill_ok if correct else fill_bad

    # Comparison pivot
    all_video = pd.concat([
        df[["video_stem", "category", "majority_prediction", "correct"]].assign(model=m)
        for m, df in model_video_dfs.items()
    ], ignore_index=True)
    pivot = all_video.pivot_table(
        index=["video_stem", "category"], columns="model",
        values="majority_prediction", aggfunc="first",
    ).reset_index()
    pivot.to_excel(writer, sheet_name="Comparison", index=False)
    _style(writer.sheets["Comparison"], pivot)

    # Summary
    summary_rows = []
    for model, df in model_video_dfs.items():
        acc      = df["correct"].mean() * 100 if not df.empty else 0
        avg_conf = df["avg_confidence"].mean() if "avg_confidence" in df.columns else None
        summary_rows.append({
            "Model":            model.upper(),
            "Videos Evaluated": len(df),
            "Correct":          int(df["correct"].sum()) if not df.empty else 0,
            "Accuracy (%)": round(acc, 1),
            "Avg Confidence":   round(float(avg_conf), 2) if avg_conf is not None and pd.notna(avg_conf) else "N/A",
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    _style(writer.sheets["Summary"], summary_df)

print(f"Workbook written: {out_xlsx}")
print()
print("=== FINAL SUMMARY ===")
print(summary_df.to_string(index=False))

# Copy all results to Drive
import shutil
DRIVE_RESULTS = Path("/content/drive/MyDrive/vlm_evaluation_results")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
for f in RESULTS_DIR.iterdir():
    shutil.copy(f, DRIVE_RESULTS / f.name)
print(f"\nAll results copied to Google Drive: {DRIVE_RESULTS}")
print("Files:", [f.name for f in DRIVE_RESULTS.iterdir()])

In [ ]:
# ── Cell 10 (optional): Download results zip to local machine ─────────────────
from google.colab import files
import zipfile

results_zip = WORK_DIR / "evaluation_results.zip"
with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in RESULTS_DIR.iterdir():
        zf.write(f, f.name)

print(f"Results zip: {results_zip.stat().st_size / 1024:.0f} KB")
files.download(str(results_zip))
print("Download started.")